# P-2026-11-18-1: Is grouped ECI "one column away" from switching axis, and does the sign problem reach the reports' sensitivity section?

**Question** (from `proposals/2026-11-17.md` #3): the 11-17 run found that dropping *Energy Drilling and Mining* from the 2022 grouped matrix gives an ECI with Spearman −0.736 against the base, even after orienting to diversity, and suggested that "the 2nd eigenvector at 17 columns picks out a different axis". Is grouped ECI one column away from switching to the λ3 axis in other years too? Since the notebook's sensitivity section (cells #158–#161) re-solves the eigenproblem for every shocked matrix, does the same instability touch the report's `[Industry 1–4]` sensitivity sentences?

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`, and every open `claude/nightly-*` branch is merged. Under PROTOCOL §1.4, tonight's question comes from `proposals/`. I took 11-17 #3 because it is cheap and diagnostic. 11-17 #1 (SDV/greedy targets without Services) overlaps Q11, which already reported the one-step target with and without Services. Q09 reported base-year eigengaps but not leave-one-column-out or shock behaviour.

- **Date:** 2026-11-18 (America/New_York), nightly run. ID `P-2026-11-18-1`.
- **Data:** `data/cleaned/2019–2023clean.csv` (lab 18 groups, re-keyed on ISO3 via `lib/research.lab_groups_iso3`. Section D reads the files in file order with every row, as the notebook's `read_country` does). `data/raw/2019–2023_mega.csv` (HS4 ECI, used only as the sign reference). `external-data/raw/wdi_panel.csv` (log GDPpc PPP). `reports/2026-spring/*.pdf` (text of §4.8 only). No downloads.
- **Years:** all of 2019–2023. Section D has to cover all five because the notebook's sensitivity cell scans 2019–2023 and prints 2023. 2023 is suspect (D1).
- **Library:** `willbetts/nightly/lib/eci.py` and `lib/research.py`, unchanged. `pypdf` is used for the report-text cell.
- **Run time:** about 9 minutes, almost all of it Section D (50,400 eigen-decompositions). The exact time is printed in the last cell.

## TL;DR

- **No axis switches. The 2022 "flip" is a sign-orientation failure.** Across all 90 leave-one-group-out drops (18 groups × 2019–2023), the dropped matrix's λ2 eigenvector never aligns more with the base λ3 than with the base λ2 (pre-stated test: **0 of 90**). Only the four *Energy Drilling and Mining* drops in 2020–2023 are weakly aligned (|r| with base v2 = **0.706–0.765**). Their negative Spearman with the base (**−0.725 / −0.801 / −0.736 / −0.758**) comes from the diversity rule picking the wrong sign. With Energy dropped, Pearson(v2, diversity) is only **0.043–0.139**, against **0.311–0.400** in the base years (B, C1). Holding M fixed and fading the Energy column out continuously never switches the axis either (min |r| 0.915 in 2022; 0 of 90 group-year paths below 0.8; B4–B5).
- **Why the diversity anchor fails:** dropping Energy re-normalises commodity exporters' shares, so they gain RCA elsewhere. In 2022, countries with Energy RCA go from mean diversity **5.41 vs 6.00** for the rest to **5.86 vs 5.52**, which reverses the diversity–complexity link the rule relies on (C2). At 18 groups, diversity is a weak anchor even in the base (margin 0.311–0.400, against 0.569–0.670 at HS4).
- **A sturdier orientation exists (post hoc, all rules reported):** taking the sign from the product side (PCI negatively correlated with ubiquity, **O4**) or from log GDPpc (**O3**) agrees with the HS4-ECI sign in **95 of 95** matrices (O1, Pearson diversity, the lib default: **91**; O2, Spearman diversity: **93**). O4's smallest margin is **0.462**, against **0.003** for O1 (C3, C4).
- **⚠️ Report impact: the notebook's sensitivity section has its own sign bug, and it reverses the `[Industry 1–4]` picks in 2019, 2022 and 2023.** Cell #158's `eci()` runs on the unfiltered CSV and takes its sign from the second row. Its base ECI is anti-oriented in **2019, 2022 and 2023** (Spearman with HS4 ECI **−0.735 / −0.753 / −0.743**; with log GDPpc **−0.494 / −0.547 / −0.529**). Within a shock curve, the sign and eigenvector never switch (0 of 50,400 shocks). So the bias is a consistent mirror: the "largest ECI gain" picks are really the industries whose expansion *lowers* complexity most. Against correctly oriented ECI, the top positive pick agrees in **0/7** report countries in each of 2019, 2022 and 2023, and in **7/7** in 2020 and 2021 (**14/35** overall). In 2023, the notebook names *Energy Drilling and Mining* as the top positive lever for Canada and the UAE, where oriented ECI names *High-complexity Products* for all 7 (D2, D4).
- **The published Spring 2026 §4.8 text shows the symptom but can't be checked:** 4 of the 6 reports where the sentence was found say "Under positive shocks, ECI **decreases** with additional investment in" their top lever (Germany, Poland, Singapore, UAE). Those runs used 2024 data that is not in the repo (D2b; data request below).


## Why this matters for the lab

This tests our **implementation**, not the economics. The reports print ECI paths, and the sensitivity section (Figures 5/6, the `[Industry 1–4]` sentences in §4.8) recomputes ECI for 80 shocked matrices per industry. If the eigenvector could jump to a different axis, or its sign could flip, under one-column changes or one-cell shocks, the printed curves and picks would be artefacts. The question also feeds research-agenda §0 link 3 (does pushing exports into a group move the index for the right reason?). Earlier nights compared grouped ECI with and without a column (Q02 aggregate, 11-17 Services). Those comparisons are only meaningful if the orientation rule they relied on is reliable.

## Method

**Pre-stated before running:**
- *Primary metric (axis switch):* for each year y ∈ 2019–2023 and each of the 18 groups g, drop g, re-apply the lab rules (drop-any-zero on the remaining 17 columns, strict RCA), and compute the λ2 eigenvector. **Switch** = its |Pearson| with the base v3 exceeds its |Pearson| with the base v2 (common countries). **Weak** = |r| with base v2 < 0.8.
- *Secondary:* Spearman of the oriented variant ECI with the base ECI, and the change in Spearman with log GDPpc on the same countries.
- *Mechanism:* is the eigengap (λ2 − λ3)/λ2 of a dropped matrix a usable warning flag? And a continuous path: weight one column by w from 1 to 0 with M held fixed, tracking λ2, λ3 and the overlaps.
- *Baseline:* the lab's grouped ECI with sorted eigenvalues, oriented to diversity (`lib/eci.eci_pci`). This is the sign-fixed version, not the notebook's raw sign (Q01).
- *Section D (sensitivity, also pre-stated):* a faithful vectorised port of cells #158/#159/#161 for the 7 Spring 2026 countries, all 18 groups, the notebook's shock grid (multipliers 1.0–3.9 in steps of 0.1, and 1.0–0.02 in steps of 0.02), for 2019–2023. Counted: shocks where column 1 of `eig` is not λ2, shocks where the notebook's sign differs from its base sign, and shocks where the shocked λ2 vector has |r| < 0.9 with the base. Then the notebook's picks (top-3 positive, top-1 negative, ranked by max gain or max loss against the base) are compared with the same picks from the true λ2 vector oriented to diversity at the base and by continuity under shocks.

**Added after seeing B (post hoc, labelled as such):** Section C, orientation rules O1–O5, scored by agreement with the sign that makes grouped ECI correlate positively with the repo's HS4 ECI (whose diversity margin is 0.57–0.67). Every rule tried is reported.


## Setup

In [1]:
# setup
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats, linalg as LA
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]
REPORT = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']
print('lib:', Path(eci.__file__).name, Path(R.__file__).name, '| numpy', np.__version__, '| pandas', pd.__version__)

lib: eci.py research.py | numpy 2.4.6 | pandas 3.1.1


Helpers. `spectrum()` applies strict RCA (the notebook's `>`) to a matrix that has already been passed through the drop-any-zero rule (`lab()`). It returns the sorted eigenvalues of $\tilde M_{cc'}$ and its first four eigenvectors. `overlap()` is the pre-stated axis test: |Pearson| of a variant's v2 with the base's v2 and with the base's v3, on common countries.

In [2]:
# helpers
def spectrum(X, k=4):
    """Lab rules already applied to X by the caller. Strict RCA, sorted eigen-decomposition of Mcc.
    Returns M, eigenvalues (desc), and the first k eigenvectors of Mcc (columns 'v1'..'vk', indexed by country).
    v2 is oriented to diversity; v3, v4 are oriented to correlate positively with diversity too (sign is arbitrary)."""
    M = eci.mcp(X, strict=True)
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    B = Mv / np.sqrt(kc[:, None] * kp[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    V = U[:, :k] / np.sqrt(kc)[:, None]
    for j in range(1, k):
        if np.corrcoef(V[:, j], kc)[0, 1] < 0:
            V[:, j] = -V[:, j]
    V = pd.DataFrame(V, index=M.index, columns=[f'v{j+1}' for j in range(k)])
    return M, S ** 2, V

def std(s):
    return (s - s.mean()) / s.std()

def overlap(var_v2, base_V):
    """|Pearson| of a variant's v2 with the base's v2 and v3, on common countries."""
    idx = var_v2.index.intersection(base_V.index)
    a = var_v2[idx]
    return abs(np.corrcoef(a, base_V.loc[idx, 'v2'])[0, 1]), abs(np.corrcoef(a, base_V.loc[idx, 'v3'])[0, 1]), len(idx)

def lab(X):
    return eci.notebook_filter(X)

G = {y: R.lab_groups_iso3(y) for y in YEARS}
LNY = {y: np.log(R.gdppc(y, 'ppp')) for y in YEARS}
BASE = {}
for y in YEARS:
    BASE[y] = spectrum(lab(G[y]))
print({y: G[y].shape for y in YEARS})

{2019: (160, 18), 2020: (154, 18), 2021: (136, 18), 2022: (141, 18), 2023: (154, 18)}


## Results

### A. Base spectra of the lab's grouped matrix

In [3]:
# A: base spectra
rows = []
for y in YEARS:
    M, lam, V = BASE[y]
    e2 = std(V.v2); e3 = std(V.v3)
    kc = M.sum(1)
    rows.append({'year': y, 'countries': len(M), 'lambda2': lam[1], 'lambda3': lam[2], 'lambda4': lam[3],
                 'gap 2-3': lam[1] - lam[2], 'relative gap (l2-l3)/l2': (lam[1] - lam[2]) / lam[1],
                 'rho(v2, diversity)': stats.spearmanr(e2, kc)[0], 'rho(v3, diversity)': stats.spearmanr(e3, kc)[0],
                 'rho(v2, ln GDPpc)': R.spearman(e2, LNY[y])[0], 'rho(v3, ln GDPpc)': R.spearman(e3, LNY[y])[0]})
A = pd.DataFrame(rows).set_index('year')
print(A.round(3).to_string())

      countries  lambda2  lambda3  lambda4  gap 2-3  relative gap (l2-l3)/l2  rho(v2, diversity)  rho(v3, diversity)  rho(v2, ln GDPpc)  rho(v3, ln GDPpc)
year                                                                                                                                                      
2019        120    0.329    0.255    0.178    0.074                    0.224               0.314               0.371              0.480             -0.289
2020        120    0.327    0.245    0.200    0.082                    0.251               0.365               0.402              0.534             -0.317
2021        105    0.336    0.269    0.211    0.067                    0.200               0.364               0.281              0.528             -0.306
2022        120    0.361    0.280    0.200    0.081                    0.224               0.425               0.484              0.554             -0.249
2023        122    0.346    0.233    0.200    0.113                   

**Takeaway (A):** the relative eigengap is 0.200–0.326, comparable to Q09's base gaps. Diversity is a weak anchor at 18 groups: ρ(v2, diversity) is only 0.314–0.425, and the λ3 vector correlates with diversity about as strongly (0.281–0.484). The λ3 vector correlates *negatively* with income (−0.249 to −0.317).

### B. Leave one group out (pre-stated)

In [4]:
# B: leave one group out
rows = []
for y in YEARS:
    Mb, lamb, Vb = BASE[y]
    base_eci = std(Vb.v2)
    rho_b_inc, _ = R.spearman(base_eci, LNY[y])
    for g in G[y].columns:
        Mv, lam, V = spectrum(lab(G[y].drop(columns=g)))
        e = std(V.v2)
        ov2, ov3, n = overlap(V.v2, Vb)
        idx = e.index.intersection(base_eci.index)
        rho_same, _ = R.spearman(e.reindex(idx), LNY[y])
        rho_base_same, _ = R.spearman(base_eci.reindex(idx), LNY[y])
        rows.append({'year': y, 'dropped': g, 'n': len(e), 'common': n,
                     'Spearman w/ base': stats.spearmanr(e[idx], base_eci[idx])[0],
                     '|r| v2 vs base v2': ov2, '|r| v2 vs base v3': ov3,
                     'switch': ov3 > ov2, 'weak (<0.8)': ov2 < 0.8,
                     'lambda2': lam[1], 'lambda3': lam[2], 'rel gap': (lam[1] - lam[2]) / lam[1],
                     'd rho income (same countries)': rho_same - rho_base_same})
B = pd.DataFrame(rows)
summ = B.groupby('year').agg(drops=('dropped', 'size'), switches=('switch', 'sum'), weak=('weak (<0.8)', 'sum'),
                             min_spearman=('Spearman w/ base', 'min'), median_spearman=('Spearman w/ base', 'median'))
summ['base rel gap'] = A['relative gap (l2-l3)/l2']
print(summ.round(3).to_string())
print('\nAll switching or weak drops:')
print(B[B.switch | B['weak (<0.8)']].round(3).to_string(index=False))
print('\nTotal switches:', int(B.switch.sum()), 'of', len(B), '| weak:', int(B['weak (<0.8)'].sum()))

      drops  switches  weak  min_spearman  median_spearman  base rel gap
year                                                                    
2019     18         0     0         0.819            0.966         0.224
2020     18         0     1        -0.725            0.973         0.251
2021     18         0     1        -0.801            0.975         0.200
2022     18         0     1        -0.736            0.982         0.224
2023     18         0     1        -0.758            0.983         0.326

All switching or weak drops:
 year                    dropped   n  common  Spearman w/ base  |r| v2 vs base v2  |r| v2 vs base v3  switch  weak (<0.8)  lambda2  lambda3  rel gap  d rho income (same countries)
 2020 Energy Drilling and Mining 120     120            -0.725              0.706              0.534   False         True    0.309    0.231    0.251                         -1.249
 2021 Energy Drilling and Mining 105     105            -0.801              0.765              0.36

**Takeaway (B):** **0 of 90** drops switch axis. 4 of 90 are weak, all *Energy Drilling and Mining* in 2020–2023, with |r| 0.706–0.765 with the base v2 (more than with v3: 0.362–0.534). Their *negative* Spearman with the base (−0.725 to −0.801) is the sign of the vector, not its axis: the absolute alignment is positive and substantial. 2019's Energy drop is ordinary (Spearman 0.819). The median drop has Spearman 0.966–0.983 with the base.

In [5]:
# B2: does the dropped matrix's own eigengap flag trouble?
r, p = stats.spearmanr(B['rel gap'], B['|r| v2 vs base v2'])
print(f'Spearman(rel gap of dropped matrix, |r| with base v2) over {len(B)} drops: {r:.3f} (p={p:.3g})')
print('rel gap of switching drops:', B.loc[B.switch, 'rel gap'].round(3).tolist())
print('rel gap quantiles of non-switching drops:', B.loc[~B.switch, 'rel gap'].quantile([0, .1, .5]).round(3).to_dict())

Spearman(rel gap of dropped matrix, |r| with base v2) over 90 drops: 0.188 (p=0.076)
rel gap of switching drops: []
rel gap quantiles of non-switching drops: {0.0: 0.164, 0.1: 0.191, 0.5: 0.249}


**Takeaway (B2):** the eigengap is not a useful warning flag. Across the 90 drops, its rank correlation with the overlap is 0.188 (p = 0.076). The Energy drops have relative gaps of 0.251–0.337, above the median of the other drops.

**Mechanism.** What do v2 and v3 represent in 2022? The table shows, for each group, the mean standardized v2 or v3 of countries with RCA in that group minus the mean of countries without it.

In [6]:
# B3: what are v2 and v3 in 2022?
y = 2022
Mb, lamb, Vb = BASE[y]
cols = Mb.columns
rows = []
for c in cols:
    m = Mb[c]
    rows.append({'group': c, 'ubiquity': int(m.sum()),
                 'mean v2 (M=1) - (M=0)': std(Vb.v2)[m == 1].mean() - std(Vb.v2)[m == 0].mean(),
                 'mean v3 (M=1) - (M=0)': std(Vb.v3)[m == 1].mean() - std(Vb.v3)[m == 0].mean()})
C1 = pd.DataFrame(rows).set_index('group').sort_values('mean v3 (M=1) - (M=0)')
print(C1.round(3).to_string())

                               ubiquity  mean v2 (M=1) - (M=0)  mean v3 (M=1) - (M=0)
group                                                                                
Precious Stones                      37                 -0.724                 -0.890
Services and Utilities               19                  1.472                 -0.817
Energy Drilling and Mining           49                 -1.050                 -0.757
Chemicals                            24                  0.555                 -0.471
High-complexity Products             26                  1.680                 -0.467
Chemical Manufacturing               34                  0.709                  0.114
Daily Instruments                    29                  1.177                  0.207
Transportation                       19                  1.364                  0.236
Weapons                              28                  0.322                  0.656
Animals and By-products              56               

**Takeaway (B3):** in 2022, v2 is the familiar complexity axis: high for High-complexity Products (+1.680), Services (+1.472) and Transportation (+1.364), and low for Energy (−1.050). v3 separates commodities and chemicals (Precious Stones −0.890, Services −0.817, Energy −0.757) from food, textiles and wood (Wood +1.081, Textile +0.950). The λ3 axis does load on Energy, but B4 shows that the axes don't trade places.

In [7]:
# B4: continuous path, Energy column weighted from 1 to 0, M held fixed
def weighted_spectrum(M, w, k=4):
    Mv = M.to_numpy(float); wv = np.asarray(w, float)
    kc = Mv @ wv; kp = Mv.sum(0)
    keep = kc > 0
    Mv, kc = Mv[keep], kc[keep]
    B_ = Mv * np.sqrt(wv / kp)[None, :] / np.sqrt(kc)[:, None]
    U, S, _ = np.linalg.svd(B_, full_matrices=False)
    V = U[:, :k] / np.sqrt(kc)[:, None]
    idx = M.index[keep]
    for j in range(1, k):
        if np.corrcoef(V[:, j], Mv.sum(1))[0, 1] < 0:
            V[:, j] = -V[:, j]
    return S ** 2, pd.DataFrame(V, index=idx, columns=[f'v{j+1}' for j in range(k)])

def path(y, col, ws=np.round(np.linspace(1, 0, 41), 3)):
    Mb, lamb, Vb = BASE[y]
    out = []
    for wv in ws:
        w = np.where(Mb.columns == col, wv, 1.0)
        lam, V = weighted_spectrum(Mb, w)
        o2, o3, _ = overlap(V.v2, Vb)
        out.append({'w': wv, 'lambda2': lam[1], 'lambda3': lam[2], 'lambda4': lam[3], 'ov base v2': o2, 'ov base v3': o3})
    return pd.DataFrame(out)

PE = path(2022, 'Energy Drilling and Mining')
print(PE.iloc[::4].round(3).to_string(index=False))
sw = PE[PE['ov base v3'] > PE['ov base v2']]
print('first weight at which v2 aligns more with base v3:', sw.w.max() if len(sw) else None)

  w  lambda2  lambda3  lambda4  ov base v2  ov base v3
1.0    0.361    0.280    0.200       1.000       0.159
0.9    0.359    0.279    0.201       0.999       0.126
0.8    0.358    0.278    0.202       0.996       0.092
0.7    0.357    0.278    0.202       0.992       0.057
0.6    0.357    0.277    0.203       0.985       0.022
0.5    0.357    0.277    0.204       0.976       0.012
0.4    0.357    0.277    0.205       0.966       0.045
0.3    0.358    0.277    0.206       0.954       0.076
0.2    0.359    0.278    0.207       0.940       0.104
0.1    0.361    0.280    0.209       0.925       0.130
0.0    0.363    0.282    0.210       0.915       0.232
first weight at which v2 aligns more with base v3: None


**Takeaway (B4):** fading the 2022 Energy column from w = 1 to 0 with M held fixed changes λ2 and λ3 by less than 0.01 and never brings them close. v2's overlap with the base v2 stays ≥ 0.915. So the 0.715 alignment of the real 17-column drop comes from **recomputing RCA** without Energy (which changes M), not from a near-degenerate eigenvalue pair.

In [8]:
# B5: same path for every (year, group): at what weight (if any) does the axis switch?
rows = []
for y in YEARS:
    for c in BASE[y][0].columns:
        P = path(y, c, ws=np.round(np.linspace(1, 0, 21), 2))
        s = P[P['ov base v3'] > P['ov base v2']]
        rows.append({'year': y, 'group': c, 'switch weight (M fixed)': s.w.max() if len(s) else np.nan,
                     'min |r| with base v2': P['ov base v2'].min()})
C3 = pd.DataFrame(rows)
print(C3.dropna().round(3).to_string(index=False))
print('pairs that switch on the M-fixed path:', int(C3['switch weight (M fixed)'].notna().sum()), 'of', len(C3))
print('pairs with min |r| < 0.8:', int((C3['min |r| with base v2'] < 0.8).sum()))

Empty DataFrame
Columns: [year, group, switch weight (M fixed), min |r| with base v2]
Index: []
pairs that switch on the M-fixed path: 0 of 90
pairs with min |r| < 0.8: 0


**Takeaway (B5):** the same M-fixed path for all 90 group-years: no switch, and no path drops below |r| = 0.8. The axis is robust. What changes under a drop is M itself.

### C. Orientation (post hoc, added after B)

In [9]:
# C1 (post hoc): is the Energy "flip" an orientation flip?
def corr_div(V, M):
    return np.corrcoef(V.v2, M.loc[V.index].sum(1))[0, 1]
rows = []
for y in YEARS:
    Mb, lamb, Vb = BASE[y]
    rows.append({'year': y, 'matrix': 'base (18 cols)', 'Pearson(v2, diversity)': corr_div(Vb, Mb), 'signed r with base v2': 1.0})
    for g in G[y].columns:
        Mv, lam, V = spectrum(lab(G[y].drop(columns=g)))
        idx = V.index.intersection(Vb.index)
        rows.append({'year': y, 'matrix': f'drop {g}', 'Pearson(v2, diversity)': corr_div(V, Mv),
                     'signed r with base v2': np.corrcoef(V.v2[idx], Vb.v2[idx])[0, 1]})
O = pd.DataFrame(rows)
print('Orientation margin |Pearson(v2, diversity)| over all 95 matrices: min %.3f, median %.3f' %
      (O['Pearson(v2, diversity)'].abs().min(), O['Pearson(v2, diversity)'].abs().median()))
print(O[O['signed r with base v2'] < 0].round(3).to_string(index=False))
print('\nBase-year orientation margins:'); print(O[O.matrix.str.startswith('base')].round(3).to_string(index=False))

Orientation margin |Pearson(v2, diversity)| over all 95 matrices: min 0.003, median 0.366
 year                          matrix  Pearson(v2, diversity)  signed r with base v2
 2020 drop Energy Drilling and Mining                   0.139                 -0.706
 2021 drop Energy Drilling and Mining                   0.061                 -0.765
 2022 drop Energy Drilling and Mining                   0.078                 -0.715
 2023 drop Energy Drilling and Mining                   0.043                 -0.748

Base-year orientation margins:
 year         matrix  Pearson(v2, diversity)  signed r with base v2
 2019 base (18 cols)                   0.311                    1.0
 2020 base (18 cols)                   0.360                    1.0
 2021 base (18 cols)                   0.389                    1.0
 2022 base (18 cols)                   0.400                    1.0
 2023 base (18 cols)                   0.400                    1.0


**Takeaway (C1):** every negative-Spearman case is an orientation failure. The diversity margin collapses to 0.043–0.139 when Energy is dropped (2020–2023), against 0.311–0.400 for the base matrices. The rule then picks the sign that is anti-correlated (−0.706 to −0.765) with the base.

In [10]:
# C2 (post hoc): what diversity does without Energy
y = 2022
X = lab(G[y]); Mb = BASE[y][0]
Md = spectrum(lab(G[y].drop(columns='Energy Drilling and Mining')))[0]
E_ = Mb['Energy Drilling and Mining']
print(f'{y}: countries with M=1 in Energy: {int(E_.sum())} of {len(E_)}')
print('mean diversity (18 cols) with/without Energy RCA: %.2f / %.2f' % (Mb.sum(1)[E_ == 1].mean(), Mb.sum(1)[E_ == 0].mean()))
print('mean diversity (17 cols, Energy dropped) same groups: %.2f / %.2f' % (Md.sum(1).reindex(E_.index)[E_ == 1].mean(), Md.sum(1).reindex(E_.index)[E_ == 0].mean()))
base_e = std(BASE[y][2].v2)
print('mean base ECI with/without Energy RCA: %.3f / %.3f' % (base_e[E_ == 1].mean(), base_e[E_ == 0].mean()))

2022: countries with M=1 in Energy: 49 of 120
mean diversity (18 cols) with/without Energy RCA: 5.41 / 6.00
mean diversity (17 cols, Energy dropped) same groups: 5.86 / 5.52
mean base ECI with/without Energy RCA: -0.621 / 0.429


**Takeaway (C2):** in 2022, the 49 countries with Energy RCA have *lower* diversity than the rest at 18 columns (5.41 vs 6.00), consistent with their low base ECI (−0.621 vs 0.429). Once Energy is dropped and RCA recomputed, their shares in everything else are inflated and they become *more* diverse (5.86 vs 5.52). The diversity–complexity link that the orientation rule relies on reverses.

Rules tried (all reported): **O1** sign of Pearson(v2, diversity) (lib default), **O2** Spearman with diversity, **O3** Spearman with log GDPpc, **O4** product side: compute the PCI direction implied by v2 and orient it to correlate negatively with ubiquity, **O5** continuity, orienting each drop to the 18-column base (only defined for variants). Reference: the sign whose grouped ECI correlates positively with the repo's HS4 ECI that year.

In [11]:
# C3 (post hoc): which orientation rule is robust? Reference = sign that correlates positively with repo HS4 ECI
H4 = {y: eci.eci_from_exports(R.repo_hs(y, 4)) for y in YEARS}   # oriented to diversity; HS4 margin printed below
for y in YEARS:
    Mh = eci.mcp(R.repo_hs(y, 4))
    print(y, 'HS4 orientation margin Pearson(ECI, diversity) = %.3f' % np.corrcoef(H4[y], Mh.loc[H4[y].index].sum(1))[0, 1])

def rule_signs(V, M, y, base_v2=None):
    """+1/-1: the sign each rule would multiply the raw v2 by (V.v2 is already diversity-oriented)."""
    v = V.v2; kc = M.loc[v.index].sum(1)
    idx_h = v.index.intersection(H4[y].index)
    out = {'O1 diversity (Pearson)': np.sign(np.corrcoef(v, kc)[0, 1]),
           'O2 diversity (Spearman)': np.sign(stats.spearmanr(v, kc)[0]),
           'O3 ln GDPpc (Spearman)': np.sign(R.spearman(v, LNY[y])[0])}
    # O4: product side. PCI from the same singular pair, oriented negatively with ubiquity; ECI takes the consistent sign
    Mv = M.loc[v.index].to_numpy(float); kp = Mv.sum(0)
    q = (Mv / kc.to_numpy()[:, None]).T @ v.to_numpy()   # PCI direction consistent with v (up to scale)
    out['O4 PCI vs ubiquity'] = -np.sign(np.corrcoef(q, kp)[0, 1])
    if base_v2 is not None:
        idx = v.index.intersection(base_v2.index)
        out['O5 continuity (vs 18-col base)'] = np.sign(np.corrcoef(v[idx], base_v2[idx])[0, 1])
    ref = np.sign(np.corrcoef(v[idx_h], H4[y][idx_h])[0, 1])
    return out, ref

rows = []
for y in YEARS:
    Mb, lamb, Vb = BASE[y]
    s, ref = rule_signs(Vb, Mb, y)
    rows.append({'year': y, 'matrix': 'base', **{k: s[k] == ref for k in s}, 'ref r': ref})
    for g in G[y].columns:
        Mv, lam, V = spectrum(lab(G[y].drop(columns=g)))
        s, ref = rule_signs(V, Mv, y, Vb.v2)
        rows.append({'year': y, 'matrix': f'drop {g}', **{k: s[k] == ref for k in s}, 'ref r': ref})
OR = pd.DataFrame(rows)
rules = [c for c in OR.columns if c.startswith('O')]
print('\nMatrices where each rule gives the sign that agrees with HS4 ECI (of 95; O5 of 90 drops):')
print(OR[rules].sum().astype(int).to_string())
print('\nDisagreements:')
print(OR.loc[~OR[rules].fillna(True).all(axis=1), ['year', 'matrix'] + rules].to_string(index=False))

2019 HS4 orientation margin Pearson(ECI, diversity) = 0.670
2020 HS4 orientation margin Pearson(ECI, diversity) = 0.622
2021 HS4 orientation margin Pearson(ECI, diversity) = 0.641
2022 HS4 orientation margin Pearson(ECI, diversity) = 0.575
2023 HS4 orientation margin Pearson(ECI, diversity) = 0.569



Matrices where each rule gives the sign that agrees with HS4 ECI (of 95; O5 of 90 drops):
O1 diversity (Pearson)            91
O2 diversity (Spearman)           93
O3 ln GDPpc (Spearman)            95
O4 PCI vs ubiquity                95
O5 continuity (vs 18-col base)    90

Disagreements:
 year                          matrix  O1 diversity (Pearson)  O2 diversity (Spearman)  O3 ln GDPpc (Spearman)  O4 PCI vs ubiquity O5 continuity (vs 18-col base)
 2020 drop Energy Drilling and Mining                   False                    False                    True                True                           True
 2021 drop Energy Drilling and Mining                   False                     True                    True                True                           True
 2022 drop Energy Drilling and Mining                   False                     True                    True                True                           True
 2023 drop Energy Drilling and Mining                   Fals

**Takeaway (C3):** O3 and O4 match the HS4 reference in **95/95** matrices, and O5 in 90/90 drops. O1 gets **91/95** and O2 **93/95**; both fail only on the Energy drops. The HS4 reference itself rests on a diversity margin of 0.569–0.670, so it is a safer anchor than the grouped diversity margin. O3 uses income, which makes it circular for any income-fit test. **O4 is the rule to adopt** for grouped variants.

In [12]:
# C4: orientation margins (how close each rule is to 0) and the Energy drop re-oriented with O4
def margins(V, M, y):
    v = V.v2; kc = M.loc[v.index].sum(1)
    Mv = M.loc[v.index].to_numpy(float); kp = Mv.sum(0)
    q = (Mv / kc.to_numpy()[:, None]).T @ v.to_numpy()
    return {'O1 |Pearson(v2, diversity)|': abs(np.corrcoef(v, kc)[0, 1]),
            'O3 |Spearman(v2, ln GDPpc)|': abs(R.spearman(v, LNY[y])[0]),
            'O4 |Pearson(PCI, ubiquity)|': abs(np.corrcoef(q, kp)[0, 1])}
mr = []
for y in YEARS:
    Mb, _, Vb = BASE[y]; mr.append({'year': y, 'matrix': 'base', **margins(Vb, Mb, y)})
    for g in G[y].columns:
        Mv, _, V = spectrum(lab(G[y].drop(columns=g))); mr.append({'year': y, 'matrix': f'drop {g}', **margins(V, Mv, y)})
MR = pd.DataFrame(mr)
print('Margin over 95 matrices (min / 5th pct / median):')
print(MR.iloc[:, 2:].quantile([0, .05, .5]).round(3).to_string())
print('\nFive smallest O1 margins:')
print(MR.nsmallest(5, 'O1 |Pearson(v2, diversity)|').round(3).to_string(index=False))

rows = []
for y in YEARS:
    Mb, _, Vb = BASE[y]; base_e = std(Vb.v2)
    Mv, _, V = spectrum(lab(G[y].drop(columns='Energy Drilling and Mining')))
    s, _ = rule_signs(V, Mv, y)
    e = std(V.v2) * s['O4 PCI vs ubiquity']
    idx = e.index.intersection(base_e.index)
    rows.append({'year': y, 'O4 sign applied': int(s['O4 PCI vs ubiquity']),
                 'Spearman w/ base (O1)': stats.spearmanr(std(V.v2)[idx], base_e[idx])[0],
                 'Spearman w/ base (O4)': stats.spearmanr(e[idx], base_e[idx])[0],
                 'rho ln GDPpc base': R.spearman(base_e[idx], LNY[y])[0], 'rho ln GDPpc drop (O4)': R.spearman(e[idx], LNY[y])[0]})
print('\nDrop Energy Drilling and Mining, re-oriented with O4:')
print(pd.DataFrame(rows).set_index('year').round(3).to_string())

Margin over 95 matrices (min / 5th pct / median):
      O1 |Pearson(v2, diversity)|  O3 |Spearman(v2, ln GDPpc)|  O4 |Pearson(PCI, ubiquity)|
0.00                        0.003                        0.411                        0.462
0.05                        0.200                        0.447                        0.511
0.50                        0.366                        0.519                        0.628

Five smallest O1 margins:
 year                          matrix  O1 |Pearson(v2, diversity)|  O3 |Spearman(v2, ln GDPpc)|  O4 |Pearson(PCI, ubiquity)|
 2019 drop Energy Drilling and Mining                        0.003                        0.664                        0.691
 2023 drop Energy Drilling and Mining                        0.043                        0.646                        0.657
 2021 drop Energy Drilling and Mining                        0.061                        0.678                        0.772
 2022 drop Energy Drilling and Mining                  

**Takeaway (C4):** O4's smallest margin over 95 matrices is **0.462** (5th percentile 0.511), against **0.003** for O1 (2019 Energy drop: right sign, by luck). Side finding, post hoc and not a recommendation: correctly oriented, the Energy-dropped grouped ECI has Spearman **0.646–0.716** with log GDPpc, against **0.480–0.554** for the 18-column base on the same countries. The 11-17 run's "−1.259 change in income correlation" was the sign error. See the avenues for how to test this without overfitting.

### D. The notebook's sensitivity section (cells #158–#161), Spring 2026 countries

In [13]:
# D: the notebook's sensitivity section (cells #158-#161), faithful port, on the 7 report countries
POS = np.arange(0, 3, 0.1)            # multipliers 1.0 ... 3.9  (percent_pos)
NEG = np.arange(1, 0, -0.02) - 1      # multipliers 1.0 ... 0.02 (percent_neg)

def nb_sens(Xv):
    """eci() of cell #158 on the whole CSV (no zero filter): strict RCA, column 1 of unsorted eig, sign from row 1.
    Also returns the true lambda2 eigenvector (sorted), standardized the same way (zero rows included)."""
    cs = Xv.sum(1); ps = Xv.sum(0)
    with np.errstate(divide='ignore', invalid='ignore'):
        M = (Xv / cs[:, None] > ps / ps.sum()).astype(float)
        kc, kp = M.sum(1), M.sum(0)
        W = np.nan_to_num(M / kc[:, None] / kp[None, :])
    mcc = np.nan_to_num(W @ M.T)
    vals, vecs = LA.eig(mcc)
    v = vecs[:, 1].real
    nb = (v - v.mean()) / v.std() if v[1] < 0 else (v.mean() - v) / v.std()
    order = np.argsort(-vals.real)
    t = vecs[:, order[1]].real
    t = (t - t.mean()) / t.std()
    return nb, t, int(order[1]) == 1, kc

SENS = {}
t0 = time.time()
for y in YEARS:
    F = eci.load_groups(y, notebook_names=False)        # file order, every row, as read_country() reads it
    iso = eci.group_name_to_iso3(F.index)
    Xv0 = F.to_numpy(float)
    nb0, t0v, ok0, kc0 = nb_sens(Xv0)
    s_div = np.sign(np.corrcoef(t0v[kc0 > 0], kc0[kc0 > 0])[0, 1])
    fx0 = t0v * s_div                                    # base, oriented to diversity
    base_sign = np.sign(np.corrcoef(nb0, fx0)[0, 1])     # +1 if the notebook's base ECI is diversity-oriented
    rows_c = {c: int(np.where(iso.values == c)[0][0]) for c in REPORT}
    for c, i in rows_c.items():
        for j, g in enumerate(F.columns):
            rec = {'nb_pos': [], 'nb_neg': [], 'fx_pos': [], 'fx_neg': [], 'col1_not_l2': 0, 'sign_change': 0, 'axis_weak': 0}
            for arr, key in ((POS, 'pos'), (NEG, 'neg')):
                for a in arr:
                    Xv = Xv0.copy(); Xv[i, j] = Xv[i, j] * (a + 1)
                    nb, t, ok, kc = nb_sens(Xv)
                    fx = t * np.sign(np.corrcoef(t, fx0)[0, 1])       # continuity orientation (O5)
                    r_axis = abs(np.corrcoef(t, fx0)[0, 1])
                    rec['col1_not_l2'] += (not ok)
                    rec['sign_change'] += np.sign(np.corrcoef(nb, fx)[0, 1]) != base_sign
                    rec['axis_weak'] += r_axis < 0.9
                    rec[f'nb_{key}'].append(nb[i]); rec[f'fx_{key}'].append(fx[i])
            SENS[(y, c, g)] = rec
    print(y, f'| notebook base ECI col1 is lambda2: {ok0} | notebook base sign vs diversity: {int(base_sign):+d} | {time.time()-t0:.0f}s')

2019 | notebook base ECI col1 is lambda2: True | notebook base sign vs diversity: -1 | 120s


2020 | notebook base ECI col1 is lambda2: True | notebook base sign vs diversity: +1 | 228s


2021 | notebook base ECI col1 is lambda2: True | notebook base sign vs diversity: +1 | 310s


2022 | notebook base ECI col1 is lambda2: True | notebook base sign vs diversity: -1 | 399s


2023 | notebook base ECI col1 is lambda2: True | notebook base sign vs diversity: -1 | 505s


**Takeaway (D):** at the base matrix, column 1 of the unsorted `eig` is λ2 in every year, but the sign rule in `eci()` (based on row 1, on the *unfiltered* CSV) is anti-oriented in **2019, 2022 and 2023**. This is a different pattern from Q01's 2021–2023 flips in `compute_eci`, which uses a different anchor row and the drop-any-zero filter. Fixing `compute_eci` alone would not fix this cell.

In [14]:
# D2: within-curve problems and the report's sensitivity picks
cnt = pd.DataFrame([{'year': y, 'country': c, 'group': g, **{k: v for k, v in r.items() if not k.startswith(('nb_', 'fx_'))}}
                    for (y, c, g), r in SENS.items()])
n_shocks = len(POS) + len(NEG)
print(f'{len(cnt)} curves x {n_shocks} shocks = {len(cnt) * n_shocks} recomputations')
print('shocks where column 1 of eig is not lambda2:', int(cnt.col1_not_l2.sum()))
print('shocks where the notebook sign differs from its own base sign:', int(cnt.sign_change.sum()))
print('shocks where |r| of shocked lambda2 vector with base < 0.9:', int(cnt.axis_weak.sum()))
print(cnt[cnt.axis_weak > 0][['year', 'country', 'group', 'axis_weak']].to_string(index=False))

def picks(y, c, kind):
    pos = {g: max(SENS[(y, c, g)][f'{kind}_pos']) - SENS[(y, c, g)][f'{kind}_pos'][0] for g in G[y].columns}
    neg = {g: SENS[(y, c, g)][f'{kind}_neg'][0] - min(SENS[(y, c, g)][f'{kind}_neg']) for g in G[y].columns}
    ps = sorted(pos, key=pos.get, reverse=True); ns = sorted(neg, key=neg.get, reverse=True)
    return ps, ns

rows = []
for y in YEARS:
    for c in REPORT:
        p_nb, n_nb = picks(y, c, 'nb'); p_fx, n_fx = picks(y, c, 'fx')
        rows.append({'year': y, 'country': c, 'nb top-3 positive': ', '.join(p_nb[:3]), 'oriented top-3 positive': ', '.join(p_fx[:3]),
                     'top-1 pos same': p_nb[0] == p_fx[0], 'top-3 pos same set': set(p_nb[:3]) == set(p_fx[:3]),
                     'top-3 overlap': len(set(p_nb[:3]) & set(p_fx[:3])),
                     'nb top negative': n_nb[0], 'oriented top negative': n_fx[0], 'top neg same': n_nb[0] == n_fx[0]})
PK = pd.DataFrame(rows)
agg = PK.groupby('year')[['top-1 pos same', 'top-3 pos same set', 'top neg same']].sum()
agg['mean top-3 overlap'] = PK.groupby('year')['top-3 overlap'].mean()
print(agg.round(2).to_string())
print('\nAll 35 country-years: top-1 positive pick agrees in', int(PK['top-1 pos same'].sum()),
      '| top-3 set agrees in', int(PK['top-3 pos same set'].sum()), '| top negative agrees in', int(PK['top neg same'].sum()))
print('\n2023 (the year the report sentence uses):')
print(PK[PK.year == 2023][['country', 'nb top-3 positive', 'oriented top-3 positive', 'nb top negative', 'oriented top negative']].to_string(index=False))

630 curves x 80 shocks = 50400 recomputations
shocks where column 1 of eig is not lambda2: 0
shocks where the notebook sign differs from its own base sign: 0
shocks where |r| of shocked lambda2 vector with base < 0.9: 43
 year country                    group  axis_weak
 2019     DEU High-complexity Products          6
 2020     DEU High-complexity Products          7
 2021     DEU High-complexity Products          2
 2022     DEU High-complexity Products          9
 2023     DEU High-complexity Products         19
      top-1 pos same  top-3 pos same set  top neg same  mean top-3 overlap
year                                                                      
2019               0                   0             0                0.14
2020               7                   7             7                3.00
2021               7                   7             7                3.00
2022               0                   0             1                0.43
2023               0         

**Takeaway (D2):** within every one of the 630 curves (50,400 shocks), column 1 stays λ2 and the notebook's sign never changes. The only axis wobble is 43 shocks, all Germany × High-complexity Products. So the curves are smooth but mirrored in the anti-oriented years. In 2019, 2022 and 2023, the notebook's top-1 positive pick matches the oriented one for **0 of 7** countries (one top-negative match in 2022). In 2020 and 2021 everything matches (7/7). In 2023, the notebook's "levers" are low-complexity groups (Energy for Canada and the UAE, Precious Stones for Germany, Singapore and the UK), while oriented ECI puts High-complexity Products first for all 7.

In [15]:
# D2b: what the published Spring 2026 reports print in section 4.8 (text only; computed on 2024 data that is not in the repo)
import re
from pypdf import PdfReader
for f in ['Canada', 'Germany', 'Poland', 'Singapore', 'United Arab Emirates', 'United Kingdom', 'Vietnam']:
    t = re.sub(r'\s+', ' ', ' '.join(pg.extract_text() or '' for pg in PdfReader(eci.REPO / f'reports/2026-spring/{f}.pdf').pages))
    i = t.find('4.8 Sensitivity Analysis Results')
    m = re.search(r'An increase in .*?\. Under positive shocks, ECI (increases|decreases) with additional investment in [^,]*', t[i:]) if i >= 0 else None
    print(f'{f}: ' + (m.group(0) if m else 'section 4.8 sentence not found'))

Canada: An increase in High-complexity Products will lead to the highest change in ECI,followed by Machinery and Construction and Precious Stones. Under positive shocks, ECI increases with additional investment in High-complexity Products


Germany: An increase in Transportation will lead to the highest change in ECI,followed by Machinery and Construction and Metals. Under positive shocks, ECI decreases with additional investment in Transportation


Poland: An increase in High-complexity Products will lead to the highest change in ECI,followed by Machinery and Construction and Metals. Under positive shocks, ECI decreases with additional investment in High-complexity Products


Singapore: An increase in Transportation will lead to the highest change in ECI,followed by High-complexity Products and Energy Drilling and Mining. Under positive shocks, ECI decreases with additional investment in Transportation


United Arab Emirates: An increase in Precious Stones will lead to the highest change in ECI, followed by Metals and Animals and By-products. Under positive shocks, ECI decreases with additional investment in Precious Stones


United Kingdom: section 4.8 sentence not found


Vietnam: An increase in Machinery and Construction will lead to the highest change in ECI,followed by Metals and Animals and By-products. Under positive shocks, ECI increases with additional investment in Machinery and Construction


**Takeaway (D2b):** this is text only. The reports say they analyse "2024", and the repo has no 2024 data (data request 2026-11-11), so their picks can't be recomputed here. Still, 4 of the 6 extracted sentences (Germany, Poland, Singapore, UAE) say ECI **decreases** under a positive shock to the industry the same sentence calls the top lever. That is what a mirrored sign produces. The UK sentence was not found by the pattern.

In [16]:
# D4: is the oriented comparator in D itself correctly signed? (unfiltered file matrix, as the sensitivity cells use)
rows = []
for y in YEARS:
    F = eci.load_groups(y, notebook_names=False)
    iso = eci.group_name_to_iso3(F.index)
    nb0, t0v, ok0, kc0 = nb_sens(F.to_numpy(float))
    fx0 = pd.Series(t0v * np.sign(np.corrcoef(t0v[kc0 > 0], kc0[kc0 > 0])[0, 1]), index=iso.values)
    fx0 = fx0[fx0.index.notna()]; fx0 = fx0[~fx0.index.duplicated()]
    nbs = pd.Series(nb0, index=iso.values); nbs = nbs[nbs.index.notna()]; nbs = nbs[~nbs.index.duplicated()]
    rows.append({'year': y, 'rows in file': len(F), 'Pearson(oriented, diversity)': np.corrcoef(t0v[kc0 > 0], kc0[kc0 > 0])[0, 1] * np.sign(np.corrcoef(t0v[kc0 > 0], kc0[kc0 > 0])[0, 1]),
                 'Spearman(oriented, HS4 ECI)': R.spearman(fx0, H4[y])[0], 'Spearman(notebook, HS4 ECI)': R.spearman(nbs, H4[y])[0],
                 'Spearman(oriented, ln GDPpc)': R.spearman(fx0, LNY[y])[0], 'Spearman(notebook, ln GDPpc)': R.spearman(nbs, LNY[y])[0]})
print(pd.DataFrame(rows).set_index('year').round(3).to_string())

      rows in file  Pearson(oriented, diversity)  Spearman(oriented, HS4 ECI)  Spearman(notebook, HS4 ECI)  Spearman(oriented, ln GDPpc)  Spearman(notebook, ln GDPpc)
year                                                                                                                                                                  
2019           161                         0.343                        0.735                       -0.735                         0.494                        -0.494
2020           155                         0.238                        0.768                        0.768                         0.562                         0.562
2021           137                         0.330                        0.829                        0.829                         0.594                         0.594
2022           142                         0.266                        0.753                       -0.753                         0.547                        -0.54

**Takeaway (D4):** the comparator is correctly signed in every year: on the unfiltered file matrix, oriented ECI has Spearman 0.735–0.829 with HS4 ECI and 0.494–0.594 with log GDPpc. The notebook's sensitivity ECI has exactly the negatives of these in 2019, 2022 and 2023.

In [17]:
# D3: figure, one report country's 2023 positive-shock curves under both conventions
c = 'VNM'; y = 2023
p_nb, _ = picks(y, c, 'nb'); p_fx, _ = picks(y, c, 'fx')
show = list(dict.fromkeys(p_nb[:3] + p_fx[:3]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, kind, title in ((axes[0], 'nb', 'notebook eci() of cell #158 (sign as computed)'), (axes[1], 'fx', 'same matrix, lambda2 oriented to diversity')):
    for g in show:
        ax.plot(POS + 1, SENS[(y, c, g)][f'{kind}_pos'], label=g)
    ax.set_title(f'{c} {y}: {title}', fontsize=9); ax.set_xlabel('shock multiplier'); ax.set_ylabel('ECI')
axes[0].legend(fontsize=7); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / 'sensitivity_vnm_2023.png', dpi=90); plt.close()
print('saved figures/sensitivity_vnm_2023.png')
print(f'\nRun time: {time.time() - T0:.0f}s')

saved figures/sensitivity_vnm_2023.png

Run time: 517s


**Takeaway (D3):** Vietnam 2023. The left panel (notebook) is the right panel mirrored. *Machinery and Construction* jumps **up** under oriented ECI and **down** in the notebook's chart, so the notebook would rank it as one of the worst levers.

![](figures/sensitivity_vnm_2023.png)

## Caveats

- **The reports' own picks are not verified.** The Spring 2026 reports used 2024 data that isn't in the repo. Section D shows what cell #158 does on the 2019–2023 repo files, and the 2024 sign could go either way. The D2b text is consistent with a mirror in at least 4 reports, but it is not proof.
- **"Oriented" means diversity-oriented at the base, then continuity.** D4 checks this comparator directly: it agrees with HS4 ECI in every year. On the filtered base matrices (C3), O1–O4 agree in all 5 years.
- **The axis test uses Pearson on raw eigenvectors**, which are not orthogonal under the plain inner product (base v2 vs base v3 is 0.159 in 2022, B4). The threshold choices (switch = v3 overlap > v2 overlap; weak < 0.8) were pre-stated but are not calibrated.
- **The Energy-drop income gain (C4) is post hoc**, measured on the same income data it would be judged on, and it is a cross-section in nominal trade.
- **2023 is suspect (D1)**, and the notebook's sensitivity sentence uses it.
- The notebook's `eci()` keeps EU-28 and zero rows, as the CSV has them. The port keeps them too (rows in file: 137–161).

## Helpful to know

- There are **two independent sign bugs** in `Spring2026.ipynb`: `compute_eci` (#14; filtered matrix, sign from row 0; flips 2021–2023 per Q01) and the sensitivity `eci()` (#158; unfiltered, sign from row 1; reversed 2019, 2022, 2023). A fix must touch both. The simplest patch for both is to orient by diversity (enough for the base matrices; margin ≥ 0.238 even unfiltered), but O4 is safer for any variant.
- `lib/eci.eci_pci` orients ECI to diversity and PCI to ubiquity **independently**. For grouped variants that drop a large column (Energy), the ECI side can be wrong while the PCI side is right. Any past result built on a grouped *variant* should be re-checked for a negative Spearman with its base. In this run, only the Energy drops were affected.
- The grouped eigenvalue pair (λ2, λ3) is not near-degenerate. Eigengap screening (Q09) is not where the fragility is.
- The notebook's sensitivity "top lever" is a max over a 1×–3.9× shock grid on step-shaped curves, so it is decided by where the RCA thresholds fall (Figure D3), not by a marginal effect.

## New research avenues (also in `proposals/2026-11-18.md`)

1. **Adopt O4 in `lib/eci.py` and re-audit past grouped variants.** Add an `orient='pci'` option, then re-run every grouped-variant comparison from Q02, Q05, 11-17 (V1/V2) and the 11-14 claim table under it, and list any sign that changes. One night.
2. **Pre-registered test of "drop commodity columns"** (C4, post hoc tonight). Pre-state the metric (Spearman with ATLAS ECI and with log GDPpc, 2019 primary), then score grouped ECI without Energy, without Energy + Precious Stones, and without Energy + Precious Stones + Metals, all O4-oriented. Then check out of sample against 2019→2023 growth (a Q10-style pilot).
3. **Draft a patched sensitivity cell** (outside `code/current`): orient the shocked ECI to diversity at the base and by continuity, report the effect at a fixed +10%/−10% shock instead of a max over 1–3.9×, and emit the `[Industry 1–4]` fields. Diff the picks against tonight's for 2019–2023.
4. **Recheck the §4.8 sentences when the 2024 file arrives** (data request 2026-11-11). Run the faithful port on 2024 for the 7 Spring 2026 countries and say which printed picks were mirrored.
5. **Germany × High-complexity shocks** (the 43 weak-axis shocks): find the multiplier where the axis wobbles and whether it changes Germany's printed curve. Low priority.

## Data issues / data needed

- No new external data is needed. Checking the published §4.8 picks needs the 2024 cleaned CSV the Spring 2026 members used. That is already requested in `data-requests/2026-11-11-report-eci-provenance.md`, and tonight's file `data-requests/2026-11-18-sensitivity-2024.md` adds this use case.
